# AI Fake Image Detector — Auto Training Notebook
**Downloads dataset directly in Colab via Kaggle API. No manual upload needed.**

Steps:
1. Enable GPU
2. Upload kaggle.json (one time only)
3. Download 140k dataset automatically
4. Train EfficientNetB0 with fine-tuning
5. Model saves to Google Drive automatically

In [ ]:
# ── CELL 1: Check GPU ─────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
    print('   Training will be fast!')
else:
    print('⚠️  No GPU! Go to: Runtime → Change runtime type → T4 GPU → Save')

In [ ]:
# ── CELL 2: Mount Google Drive (model saves here automatically) 
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_PATH = '/content/drive/MyDrive/model.h5'
print(f'✅ Drive mounted! Model will save to: {OUTPUT_PATH}')

In [ ]:
# ── CELL 3: Install Kaggle and upload kaggle.json ─────────────
# Get kaggle.json from: kaggle.com → Your Profile → Settings → API → Create New Token
!pip install -q kaggle

from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API configured!')

In [ ]:
# ── CELL 4: Download dataset directly in Colab ────────────────
# This downloads 140k real and fake face images (~1.2GB)
# Takes ~3-5 minutes depending on Colab's connection speed

print('Downloading 140k Real and Fake Faces dataset...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
print('Extracting...')
!unzip -q 140k-real-and-fake-faces.zip
print('✅ Dataset downloaded and extracted!')

# Check what we got
import os
base = 'real_vs_fake/real-vs-fake'
for split in ['train', 'valid', 'test']:
    for label in ['real', 'fake']:
        path = f'{base}/{split}/{label}'
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f'  {split}/{label}: {count:,} images')

In [ ]:
# ── CELL 5: Merge train/valid/test into one dataset/ folder ───
# Our app.py expects dataset/real/ and dataset/fake/
# This merges all 3 splits into one folder for training

import shutil
import os

os.makedirs('dataset/real', exist_ok=True)
os.makedirs('dataset/fake', exist_ok=True)

base = 'real_vs_fake/real-vs-fake'
total = {'real': 0, 'fake': 0}

for split in ['train', 'valid', 'test']:
    for label in ['real', 'fake']:
        src = f'{base}/{split}/{label}'
        if not os.path.exists(src):
            continue
        files_list = os.listdir(src)
        print(f'Copying {len(files_list):,} {label} images from {split}...')
        for fname in files_list:
            src_file = os.path.join(src, fname)
            dst_file = os.path.join('dataset', label, f'{split}_{fname}')
            if not os.path.exists(dst_file):
                shutil.copy2(src_file, dst_file)
                total[label] += 1

print(f'\n✅ Dataset merged!')
print(f'   Real : {total["real"]:,}')
print(f'   Fake : {total["fake"]:,}')
print(f'   Total: {sum(total.values()):,}')

In [ ]:
# ── CELL 6: Full Training — EfficientNetB0 with fine-tuning ───
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS_1   = 10
EPOCHS_2   = 10
DATASET_DIR = 'dataset'

# ── Class weights ─────────────────────────────────────────────
exts = ('.jpg', '.jpeg', '.png', '.webp')
real_count = len([f for f in os.listdir('dataset/real') if f.lower().endswith(exts)])
fake_count = len([f for f in os.listdir('dataset/fake') if f.lower().endswith(exts)])
total = real_count + fake_count
weight_real = total / (2 * real_count)
weight_fake = total / (2 * fake_count)
print(f'Real: {real_count:,} | Fake: {fake_count:,} | Total: {total:,}')

# ── Data generators with augmentation ────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    width_shift_range=0.1,
    height_shift_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False
)

# Map class weights
class_weight = {}
for cls_name, idx in train_gen.class_indices.items():
    class_weight[idx] = weight_real if cls_name == 'real' else weight_fake
print(f'Class mapping : {train_gen.class_indices}')
print(f'Class weights : {class_weight}')

# ── Build EfficientNetB0 model ────────────────────────────────
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(1, activation='sigmoid')(x)

model = Model(base.input, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── Phase 1: Train top layers only ───────────────────────────
print('\nPhase 1 — Training top layers...')
callbacks1 = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    ModelCheckpoint(OUTPUT_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]
history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_1, class_weight=class_weight, callbacks=callbacks1
)
print(f'Phase 1 best: {max(history1.history["val_accuracy"])*100:.1f}%')

# ── Phase 2: Fine-tune last 20 layers ────────────────────────
print('\nPhase 2 — Fine-tuning last 20 layers...')
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
callbacks2 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(OUTPUT_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, verbose=1)
]
history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_2, class_weight=class_weight, callbacks=callbacks2
)

# ── Final results ─────────────────────────────────────────────
best = max(history1.history['val_accuracy'] + history2.history['val_accuracy'])
size_mb = os.path.getsize(OUTPUT_PATH) / (1024*1024)
print('\n' + '='*50)
print('TRAINING COMPLETE')
print('='*50)
print(f'Best accuracy     : {best*100:.1f}%')
print(f'Model size        : {size_mb:.1f} MB')
print(f'Saved directly to : {OUTPUT_PATH}')
if best >= 0.90:
    print('✅ Excellent! Model is ready to use.')
elif best >= 0.75:
    print('⚠️  Decent — may still misclassify some edge cases.')
else:
    print('❌ Low accuracy — try more epochs or check dataset.')

In [ ]:
# ── CELL 7: Test model right here in Colab ────────────────────
from google.colab import files as colab_files
from tensorflow.keras.models import load_model
from PIL import Image
import numpy as np

print('Loading model from Google Drive...')
model = load_model(OUTPUT_PATH)
print('✅ Model loaded! Upload any image to test:')

uploaded = colab_files.upload()

for fname, data in uploaded.items():
    with open(fname, 'wb') as f:
        f.write(data)
    img = Image.open(fname).convert('RGB').resize((224, 224))
    arr = np.expand_dims(np.array(img) / 255.0, axis=0)
    pred = model.predict(arr, verbose=0)[0][0]
    label = 'FAKE' if pred > 0.5 else 'REAL'
    confidence = pred * 100 if pred > 0.5 else (1 - pred) * 100
    print(f'\nFile       : {fname}')
    print(f'Result     : {label}')
    print(f'Confidence : {confidence:.1f}%')
    print(f'Raw score  : {pred:.4f}  (>0.5 = Fake, <0.5 = Real)')

In [ ]:
# ── CELL 8: Confirm model is saved on Drive ───────────────────
import os
if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024*1024)
    print(f'✅ model.h5 saved on Google Drive')
    print(f'   Path : {OUTPUT_PATH}')
    print(f'   Size : {size_mb:.1f} MB')
    print()
    print('To use on your PC:')
    print('  1. Open drive.google.com')
    print('  2. Find model.h5 in My Drive')
    print('  3. Right-click → Download')
    print('  4. Replace your backend/model.h5')
    print('  5. Restart Flask: python app.py')
else:
    print(f'❌ model.h5 not found. Run Cell 6 first.')